# DSAI 413 — A2: Notebook 3 — QA Pipeline (Mode 2)

**Goal:** Build the RAG-based QA pipeline end-to-end.

**Steps:**
1. Build the QA dataset from MIMIC-CXR reports
2. Build the FAISS index using ColPali embeddings
3. Run the full QA pipeline on sample queries
4. Evaluate with Exact Match + Token F1
5. Visualize retrieved contexts and scores

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.config import (
    MIMIC_CSV_PATH, MIMIC_IMG_COL, MIMIC_TEXT_COL,
    DATASET_SUBSET_SIZE, QA_PAIRS_PATH, FAISS_TOP_K
)
from src.preprocessing import load_mimic_subset
from src.qa_dataset_creation import build_qa_dataset
from src.retrieval import build_index_from_images
from src.mode2_qa import QAPipeline
from src.evaluation import compute_qa_metrics

print('Imports OK')

## 1. Load data

In [ ]:
images, reports, img_paths = load_mimic_subset(
    MIMIC_CSV_PATH,
    img_col=MIMIC_IMG_COL,
    text_col=MIMIC_TEXT_COL,
    subset_size=DATASET_SUBSET_SIZE,
)
print(f'Loaded {len(images)} images')

## 2. Build QA dataset (skip if already done)

In [ ]:
import os
if not os.path.exists(QA_PAIRS_PATH):
    print('Building QA dataset…')
    qa_pairs = build_qa_dataset(
        reports=reports,
        image_paths=img_paths,
        use_groq=True,
    )
else:
    with open(QA_PAIRS_PATH) as f:
        qa_pairs = json.load(f)
    print(f'Loaded existing QA dataset: {len(qa_pairs)} pairs')

# Preview
for pair in qa_pairs[:3]:
    print(f'  Q: {pair["question"]}')
    print(f'  A: {pair["answer"]}')
    print(f'  Category: {pair["category"]} | Method: {pair["generation_method"]}')
    print()

## 3. Build ColPali FAISS index (skip if already built)

In [ ]:
from src.config import FAISS_INDEX_PATH

if not os.path.exists(FAISS_INDEX_PATH):
    print('Building FAISS index with ColPali…')
    # Use training split for the index
    with open('../data/split_indices.json') as f:
        splits = json.load(f)
    train_idx = splits['train']

    train_images  = [images[i]    for i in train_idx]
    train_reports = [reports[i]   for i in train_idx]
    train_paths   = [img_paths[i] for i in train_idx]

    index = build_index_from_images(
        images=train_images,
        reports=train_reports,
        image_paths=train_paths,
        backend='colpali',
        save=True,
    )
    print(f'Index saved: {index.index.ntotal} vectors')
else:
    from src.retrieval import RetrievalIndex
    index = RetrievalIndex.load()
    print(f'Index loaded: {index.index.ntotal} vectors')

## 4. Run QA pipeline on sample queries

In [ ]:
# Load the QA pipeline
qa_pipeline = QAPipeline(index=index, retrieval_backend='colpali', top_k=FAISS_TOP_K)
qa_pipeline.load_models()

# Run on first 3 QA pairs
for pair in qa_pairs[:3]:
    try:
        query_image = __import__('src.preprocessing', fromlist=['load_image']).load_image(pair['image_path'])
        result = qa_pipeline.run(
            image=query_image,
            question=pair['question'],
            ground_truth_answer=pair['answer'],
        )

        print(f"Q: {result['question']}")
        print(f"A (model): {result['answer']}")
        print(f"A (GT):    {pair['answer']}")
        print(f"Metrics:   {result['metrics']}")
        print(f"Top retrieved score: {result['retrieved_results'][0]['score']:.4f}\n")

    except Exception as e:
        print(f'Error: {e}')

## 5. Batch evaluation

In [ ]:
from src.preprocessing import load_image

eval_pairs = qa_pairs[:50]  # evaluate on first 50
predictions, ground_truths = [], []

for pair in eval_pairs:
    try:
        img = load_image(pair['image_path'])
        result = qa_pipeline.run(image=img, question=pair['question'])
        predictions.append(result['answer'])
        ground_truths.append(pair['answer'])
    except Exception:
        pass

qa_metrics = compute_qa_metrics(predictions, ground_truths)
print('\n=== Mode 2 QA Evaluation ===')
for k, v in qa_metrics.items():
    print(f'  {k}: {v}')

## 6. Visualize retrieval scores for a single query

In [ ]:
pair = qa_pairs[0]
query_image = load_image(pair['image_path'])
result = qa_pipeline.run(image=query_image, question=pair['question'])

retrieved = result['retrieved_results']
ranks  = [f"Rank {i+1}" for i in range(len(retrieved))]
scores = [r['score'] for r in retrieved]

plt.figure(figsize=(8, 4))
bars = plt.bar(ranks, scores, color='teal', edgecolor='white')
plt.title(f'ColPali Retrieval Scores\nQuery: "{pair["question"]}"')
plt.ylabel('Cosine Similarity Score')
plt.ylim(0, 1)
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{score:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nAnswer: {result["answer"]}')
print(f'\nRetrieved Context (top result snippet):')
top_report = retrieved[0]['report']
print(top_report[:300] + '…' if len(top_report) > 300 else top_report)